# AWS Bedrock AgentCore - Filesystem 구성을 통한 Managed Session Storage

이 Notebook에서는 다음 방법을 살펴봅니다.
1. filesystem 구성을 사용하는 Bedrock AgentCore agent 생성 및 배포
2. 표준 prompt로 에이전트를 호출하여 filesystem에 session 관련 정보 생성
3. [`invoke_agent_runtime_command`](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore/client/invoke_agent_runtime_command.html)를 사용하여 agent runtime에서 system command를 직접 실행하고 filesystem의 정보 확인
4. 다른 session에서 command를 다시 실행하여 session isolation 확인

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                 |
|:--------------------|:----------------------------------------------------------|
| 튜토리얼 유형       | Runtime의 Managed Session Storage                         |
| Tool 유형           | HTTP server                                               |
| 튜토리얼 구성 요소  | AgentCore Runtime에 호스팅                                |
| 튜토리얼 분야       | 산업 공통                                                 |
| 예제 난이도         | 중간                                                       |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 boto3              |


### 튜토리얼 아키텍처
이 튜토리얼 Notebook에서는 에이전트 하나를 빌드합니다. 먼저 `persistent-notes` 폴더의 Claude skill과 함께 에이전트를 AgentCore Runtime에 배포합니다. 그런 다음 에이전트를 호출하여 로컬 note를 생성하고 로컬 파일을 확인합니다. 마지막으로 session isolation이 실제로 작동하는 모습을 확인합니다.

이제 시작하겠습니다.

## 1단계: Dependency 설치

먼저 boto3와 Bedrock AgentCore SDK를 포함한 필수 Python package를 설치합니다.

In [ ]:
!uv pip install -qU -r requirements.txt

In [ ]:
!uv pip freeze | grep boto3

**계속하기 전에 kernel을 재시작하세요.**

## 2단계: IAM Execution Role 생성

AgentCore Runtime에 필요한 권한을 가진 IAM role을 생성합니다.
- container image를 가져오기 위한 ECR 액세스
- logging을 위한 CloudWatch Logs
- Bedrock 모델 호출

In [ ]:
# IAM ROLE 생성
from helpers.utils import create_agentcore_runtime_execution_role, SAMPLE_ROLE_NAME

execution_role_arn = create_agentcore_runtime_execution_role(SAMPLE_ROLE_NAME)
execution_role_arn

---

## 3단계: 배포

더 나은 dependency 관리와 일관성을 위해 Docker 배포를 사용합니다.

### 배포 수행(Docker 빌드 및 ECR에 push)

이제 Docker image를 빌드하여 ECR에 push합니다. 먼저 repository가 있는지 확인하고, 없으면 생성합니다.

#### AWS Client 설정

boto3 client를 초기화하고 ECR 작업에 필요한 AWS 계정 정보를 가져옵니다.

In [ ]:
import json
import boto3
import subprocess
import base64
from boto3.session import Session

boto_session = Session()
sts = boto3.client("sts")

account_id = sts.get_caller_identity()["Account"]
region = boto_session.region_name
region

In [ ]:
repo_name = "managed-session-agent-demo"
ecr = boto3.client("ecr", region_name=region)

try:
    response = ecr.create_repository(repositoryName=repo_name)
    repo_uri = response["repository"]["repositoryUri"]
    repo_arn = response["repository"]["repositoryArn"]
    print(f"✅ Created repository: {repo_uri}")
except ecr.exceptions.RepositoryAlreadyExistsException:
    print("ℹ️ Repository already exists")
    response = ecr.describe_repositories(repositoryNames=[repo_name])
    repo_uri = response["repositories"][0]["repositoryUri"]
    repo_arn = response["repositories"][0]["repositoryArn"]

print(f"Repository URI: {repo_uri}")
print(f"Repository ARN: {repo_arn}")

#### ECR 인증

ECR에서 authorization token을 가져와 Docker에 login합니다. 그러면 private ECR repository에 image를 push할 수 있습니다.

In [ ]:
auth = ecr.get_authorization_token()
token = base64.b64decode(auth["authorizationData"][0]["authorizationToken"]).decode().split(":")[1]
subprocess.run(
    f"echo {token} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com",
    shell=True,
)

#### Docker Image 빌드

현재 디렉터리의 Dockerfile을 사용하여 로컬에서 Docker image를 빌드합니다. 이 과정에서 에이전트 코드와 dependency를 package로 만듭니다.

In [ ]:
docker_build = subprocess.run(["docker", "build", "-t", f"{repo_name}:latest", "."])

#### Docker Image tag 지정

로컬 image를 ECR에 push할 수 있도록 ECR repository URI로 tag를 지정합니다.

In [ ]:
subprocess.run(["docker", "tag", f"{repo_name}:latest", f"{repo_uri}:latest"])

#### ECR에 push

tag가 지정된 image를 ECR에 push합니다. 그러면 AgentCore Runtime에서 이 image를 가져와 배포할 수 있습니다.

Docker push 실행

In [ ]:
subprocess.run(["docker", "push", f"{repo_uri}:latest"])

print(f"✅ Pushed to: {repo_uri}:latest")

---

## 4단계: AgentCore Runtime 생성

이제 다음 설정으로 agent runtime을 생성합니다.
- ECR image를 가리키는 container 구성
- 권한을 위한 IAM role
- `/mnt/workspace`에 `sessionStorage`가 mount된 **Filesystem 구성**

이 filesystem 구성은 session isolation을 활성화하여 각 session에 격리된 storage를 제공합니다.

### 에이전트 생성

새로운 Managed Session Storage 기능을 지원하는 에이전트를 생성합니다.

In [ ]:
acc_runtime = boto3.client(
    "bedrock-agentcore-control",
    region_name=region,
)

ac_name = "managed_session_agent_demo"

In [ ]:
response = acc_runtime.create_agent_runtime(
    agentRuntimeName=ac_name,
    agentRuntimeArtifact={"containerConfiguration": {"containerUri": f"{repo_uri}:latest"}},
    roleArn=execution_role_arn,
    protocolConfiguration={"serverProtocol": "HTTP"},
    networkConfiguration={"networkMode": "PUBLIC"},
    filesystemConfigurations=[{"sessionStorage": {"mountPath": "/mnt/workspace"}}],
)

이미 생성된 Runtime을 업데이트하려면 다음 셀의 주석을 해제합니다.

In [ ]:
# response = acc_runtime.update_agent_runtime(
#     agentRuntimeId=agent_id,
#     agentRuntimeArtifact={
#         'containerConfiguration': {
#             'containerUri': f'{repo_uri}:latest'
#         }
#     },
#     roleArn=execution_role_arn,
#     protocolConfiguration={
#         'serverProtocol': 'HTTP'
#     },
#     networkConfiguration={
#         'networkMode': 'PUBLIC'
#     },
#     filesystemConfigurations=[{
#         'sessionStorage':{
#             "mountPath": "/mnt/workspace"
#         }
#     }]
# )

In [ ]:
agent_arn = response["agentRuntimeArn"]
agent_id = response["agentRuntimeId"]
agent_arn, agent_id

#### 에이전트 상태 확인

agent runtime이 성공적으로 생성되어 READY 상태인지 확인합니다.

In [ ]:
response = acc_runtime.get_agent_runtime(agentRuntimeId=agent_id)
response["status"]

### 에이전트 테스트

먼저 에이전트를 호출할 AgentCore client를 생성합니다.

In [ ]:
agentcore_client = boto3.client(
    "bedrock-agentcore",
    region_name=region,
)

#### 첫 번째 에이전트 호출

reminder를 저장하도록 에이전트를 호출합니다. 에이전트는 다음 작업을 수행합니다.
1. `persistent-notes` skill 사용
2. note를 `/mnt/workspace/notes.json`에 저장
3. 이 session을 식별하는 `runtimeSessionId` 반환

In [ ]:
response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "save a reminder for tmr 9am to call my brother."}),
)

# command output stream 처리
for event in response["response"].iter_lines():
    if event:
        line = event.decode("utf-8")
        if line.startswith("data: "):
            data = json.loads(line[6:])
            if "content" in data:
                for item in data["content"]:
                    if "text" in item:
                        print(item["text"])

In [ ]:
# lifecycle 관리를 위해 runtime session ID 저장
runtime_session_id = response.get("runtimeSessionId")
print(f"Runtime Session ID: {runtime_session_id}")

이 session이 종료되었는지 확인하기 위해 session을 중지합니다.

In [ ]:
response = agentcore_client.stop_runtime_session(runtimeSessionId=runtime_session_id, agentRuntimeArn=agent_arn)
response

이제 동일한 session ID로 새 session을 시작하고 새로운 note를 요청합니다. `.json` 파일에는 note가 두 개 있어야 합니다.

In [ ]:
response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    runtimeSessionId=runtime_session_id,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "save a reminder for sunday 11am to have family lunch."}),
)

# command output stream 처리
for event in response["response"].iter_lines():
    if event:
        line = event.decode("utf-8")
        if line.startswith("data: "):
            data = json.loads(line[6:])
            if "content" in data:
                for item in data["content"]:
                    if "text" in item:
                        print(item["text"])

# lifecycle 관리를 위해 runtime session ID 저장
runtime_session_id = response.get("runtimeSessionId")
print(f"Runtime Session ID: {runtime_session_id}")

#### Session Storage 검사

`invoke_agent_runtime_command`를 사용하여 agent runtime container에서 shell command를 직접 실행합니다.

이 command는 `/mnt/workspace` session storage에서 notes 파일을 읽어 note가 유지되었음을 보여줍니다.

In [ ]:
# agent runtime에서 system command 실행
# Command: skill에서 생성한 /mnt/workspace/notes.json 파일 출력
response = agentcore_client.invoke_agent_runtime_command(
    agentRuntimeArn=agent_arn,
    runtimeSessionId=runtime_session_id,
    body={
        "command": '/bin/bash -c "cat /mnt/workspace/notes.json"',  # 실행할 shell command
        "timeout": 300,  # timeout 초 단위(5분)
    },
)

# command output stream 처리
for event in response["stream"]:
    if "chunk" in event:
        chunk = event["chunk"]
        print(chunk)

#### 두 번째 Session 생성

다른 prompt로 에이전트를 다시 호출합니다. 그러면 자체 격리 storage를 갖는 **새 session**이 생성됩니다.
이 과정에서 **session isolation 기능**도 확인할 수 있습니다.

In [ ]:
response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "remind me to wash my car on saturday."}),
)

for event in response["response"].iter_lines():
    if event:
        line = event.decode("utf-8")
        if line.startswith("data: "):
            data = json.loads(line[6:])
            if "content" in data:
                for item in data["content"]:
                    if "text" in item:
                        print(item["text"])

# lifecycle 관리를 위해 runtime session ID 저장
runtime_session_id = response.get("runtimeSessionId")
print(f"Runtime Session ID: {runtime_session_id}")

#### Session Isolation 확인

두 번째 session에서 동일한 command를 실행합니다. 다음 사항을 확인할 수 있습니다.
- 각 session에 자체 `/mnt/workspace/notes.json`이 있음
- note가 session 간에 격리됨
- Session 1에는 형제에게 전화하라는 reminder만 표시됨
- Session 2에는 세차하라는 reminder만 표시됨

이를 통해 **Managed Session Storage**가 작동하여 AgentCore가 session별 storage를 자동으로 격리하는 것을 확인할 수 있습니다.

In [ ]:
# agent runtime에서 system command 실행
# Command: skill에서 생성한 /mnt/workspace/notes.json 파일 출력
response = agentcore_client.invoke_agent_runtime_command(
    agentRuntimeArn=agent_arn,
    runtimeSessionId=runtime_session_id,
    body={
        "command": '/bin/bash -c "cat /mnt/workspace/notes.json"',  # 실행할 shell command
        "timeout": 300,  # timeout 초 단위(5분)
    },
)

# command output stream 처리
for event in response["stream"]:
    if "chunk" in event:
        chunk = event["chunk"]
        print(chunk)

두 호출 모두에서 `runtimeSessionId`를 사용하며, 기록된 event가 격리되어 각각의 session과 연결된 것을 확인할 수 있습니다.

---

## 5단계: 리소스 정리(선택 사항)

Runtime 삭제

In [ ]:
acc_runtime.delete_agent_runtime(agentRuntimeId=agent_id)

In [ ]:
from helpers.utils import delete_agentcore_runtime_execution_role, SAMPLE_ROLE_NAME

delete_agentcore_runtime_execution_role(SAMPLE_ROLE_NAME)